<a href="https://colab.research.google.com/github/surajgade17/csv-chemical-parser/blob/main/CSV_Chemical_Parser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSV Chemical Parser

**Author:** Suraj Gade

**GitHub:** github.com/surajgade17

**Phase:** 1 - Python Foundation

## What This Project Does
This project reads chemical molecule data validates SMILES strings using RDKit, extracts molecular properties and saves cleaned data to a CSV file.

## Tool Used
- Python
- Pandas
- RDKit

In [55]:
!pip install rdkit

In [56]:
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
RDLogger.DisableLog('rdApp.*')

## 2 - Create Sample Dataset

We create a chemical dataset containing :
- Molecule names
- SMILES strings
- Solubility values

Dataset has 5 valid + 1 invalid molecule

In [57]:
data = {
    'Name':['Ethanol','Aspirin','Benzene','Caffne','Ibuprofen','BadMolecule'],
    'SMILES':['CCO','CC(=O)Oc1ccccc1C(=O)O','c1ccccc1','CN1C=NC2=C1C(=O)N(C(=O)N2C)C','CC(C)Cc1ccc(cc1)C(C)C(=O)O','INVALID_SMILES'],
    'Solubility':[-0.24,-1.10,-1.85,-0.07,-3.70,-2.50]
}
df = pd.DataFrame(data)
print("ORIGINAL DATASET:")
print(df)

ORIGINAL DATASET:
          Name                        SMILES  Solubility
0      Ethanol                           CCO       -0.24
1      Aspirin         CC(=O)Oc1ccccc1C(=O)O       -1.10
2      Benzene                      c1ccccc1       -1.85
3       Caffne  CN1C=NC2=C1C(=O)N(C(=O)N2C)C       -0.07
4    Ibuprofen    CC(C)Cc1ccc(cc1)C(C)C(=O)O       -3.70
5  BadMolecule                INVALID_SMILES       -2.50


## 3 Validate SMILES Function

This function :
- Takes one SMILES string as input
- Builds molecule using RDKit
- If valid → returns MW, Formula, LogP, Atoms
- If invalid → returns None values

In [58]:
def parse_smiles (smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return{
            'Valid':False,
            'Molecular Weight':None,
            'Formula':None,
            'LogP':None,
            'Atoms':None,
        }
    return{
        'Valid':True,
        'Molecular Weight':round(Descriptors.MolWt(mol), 2),
        'Formula':Chem.rdMolDescriptors.CalcMolFormula(mol),
        'LogP':round(Descriptors.MolLogP(mol), 2),
        'Atoms':mol.GetNumAtoms(),
    }

## 4 Process All Molecule

We loop through wach SMILES in DataFrame 4:
- Apply parse_smiles to each molecule
- Store result in list
- Add list as new column to DataFrame


In [59]:
valid_list = []
mw_list = []
formula_list = []
logp_list = []
atoms_list = []

for smiles in df['SMILES']:
    result = parse_smiles(smiles)
    valid_list.append(result['Valid'])
    mw_list.append(result['Molecular Weight'])
    formula_list.append(result['Formula'])
    logp_list.append(result['LogP'])
    atoms_list.append(result['Atoms'])

df['Valid'] = valid_list
df['Molecular Weight'] = mw_list
df['Formula'] = formula_list
df['LogP'] = logp_list
df['Atoms'] = atoms_list

print("PROCESSED DATASET:")
print(df)

PROCESSED DATASET:
          Name                        SMILES  Solubility  Valid  \
0      Ethanol                           CCO       -0.24   True   
1      Aspirin         CC(=O)Oc1ccccc1C(=O)O       -1.10   True   
2      Benzene                      c1ccccc1       -1.85   True   
3       Caffne  CN1C=NC2=C1C(=O)N(C(=O)N2C)C       -0.07   True   
4    Ibuprofen    CC(C)Cc1ccc(cc1)C(C)C(=O)O       -3.70   True   
5  BadMolecule                INVALID_SMILES       -2.50  False   

   Molecular Weight    Formula  LogP  Atoms  
0             46.07      C2H6O -0.00    3.0  
1            180.16     C9H8O4  1.31   13.0  
2             78.11       C6H6  1.69    6.0  
3            194.19  C8H10N4O2 -1.03   14.0  
4            206.28   C13H18O2  3.07   15.0  
5               NaN       None   NaN    NaN  


## 5 - Clean Data

Separate molecules in two groups:
- valild_df → molecule RDKit accepted
- invalid_df → molecule RDKit rejected

In [60]:

valid_df = df[df['Valid'] == True].copy()

invalid_df = df[df['Valid'] == False].copy()

print("VALID MOLECULES:")
print(valid_df)

print("\nINVALID MOLECULES:")
print(invalid_df)

print(f"\nTotal    : {len(df)}")
print(f"Valid    : {len(valid_df)}")
print(f"Invalid  : {len(invalid_df)}")

VALID MOLECULES:
        Name                        SMILES  Solubility  Valid  \
0    Ethanol                           CCO       -0.24   True   
1    Aspirin         CC(=O)Oc1ccccc1C(=O)O       -1.10   True   
2    Benzene                      c1ccccc1       -1.85   True   
3     Caffne  CN1C=NC2=C1C(=O)N(C(=O)N2C)C       -0.07   True   
4  Ibuprofen    CC(C)Cc1ccc(cc1)C(C)C(=O)O       -3.70   True   

   Molecular Weight    Formula  LogP  Atoms  
0             46.07      C2H6O -0.00    3.0  
1            180.16     C9H8O4  1.31   13.0  
2             78.11       C6H6  1.69    6.0  
3            194.19  C8H10N4O2 -1.03   14.0  
4            206.28   C13H18O2  3.07   15.0  

INVALID MOLECULES:
          Name          SMILES  Solubility  Valid  Molecular Weight Formula  \
5  BadMolecule  INVALID_SMILES        -2.5  False               NaN    None   

   LogP  Atoms  
5   NaN    NaN  

Total    : 6
Valid    : 5
Invalid  : 1


## 6 - Save to CSV

Save cleaned valid molecule to CSV file :
- File name : cleaned_molecule.csv
- Read back to verify file saved correctly

In [61]:
valid_df.to_csv('cleaned_molecules.csv', index=False)
print("FILE SAVED: cleaned_molecules.csv")

saved_df = pd.read_csv('cleaned_molecules.csv')
print("\nVERIFICATION:")
print(saved_df)

FILE SAVED: cleaned_molecules.csv

VERIFICATION:
        Name                        SMILES  Solubility  Valid  \
0    Ethanol                           CCO       -0.24   True   
1    Aspirin         CC(=O)Oc1ccccc1C(=O)O       -1.10   True   
2    Benzene                      c1ccccc1       -1.85   True   
3     Caffne  CN1C=NC2=C1C(=O)N(C(=O)N2C)C       -0.07   True   
4  Ibuprofen    CC(C)Cc1ccc(cc1)C(C)C(=O)O       -3.70   True   

   Molecular Weight    Formula  LogP  Atoms  
0             46.07      C2H6O -0.00    3.0  
1            180.16     C9H8O4  1.31   13.0  
2             78.11       C6H6  1.69    6.0  
3            194.19  C8H10N4O2 -1.03   14.0  
4            206.28   C13H18O2  3.07   15.0  


## Summary
Final statistics of processed dataset.

In [62]:
print("=" * 40)
print("CSV CHEMICAL PARSER — Suraj Gade")
print("=" * 40)
print(f"Total    : {len(df)}")
print(f"Valid    : {len(valid_df)}")
print(f"Invalid  : {len(invalid_df)}")
print(f"Avg MW   : {round(valid_df['Molecular Weight'].mean(), 2)}")
print(f"Avg LogP : {round(valid_df['LogP'].mean(), 2)}")
print("=" * 40)
print("Complete! Output: cleaned_molecules.csv")
print("=" * 40)

CSV CHEMICAL PARSER — Suraj Gade
Total    : 6
Valid    : 5
Invalid  : 1
Avg MW   : 140.96
Avg LogP : 1.01
Complete! Output: cleaned_molecules.csv
